# Dataset B — XGBoost baseline (XGB-01)

Notebook này triển khai đúng một baseline XGBoost cố định trên năm fold Train đã preprocess. Không đọc Validation/Test, không preprocess lại, không tuning, không weighting/resampling và không early stopping.

## 1. XGB-00/XGB-01 preflight — khóa môi trường và Train folds\n
\n
Cell này fail-fast nếu environment, dependency, protocol, feature/split version, fold membership, ID/label hoặc matrix không khớp contract. Chênh lệch giữa phiên bản runtime và phiên bản đã tạo preprocessing artifact được hiển thị riêng, không bị mô tả nhầm là cùng environment.\n

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
import yaml
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "config/dataset_b/model_protocol_v1.yaml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the Dataset B project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if Path(sys.prefix).name.lower() != "credit-risk":
    raise RuntimeError("Select the credit-risk Python environment before running XGB-01.")
if xgboost.__version__ != "3.2.0":
    raise RuntimeError(
        f"XGB-01 requires xgboost==3.2.0; found {xgboost.__version__}."
    )

from src.dataset_b.preprocessing import DatasetBSplitLoader

runtime_versions = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
}

config_dir = PROJECT_ROOT / "config/dataset_b"
protocol_path = config_dir / "model_protocol_v1.yaml"
feature_path = config_dir / "feature_set_v1.yaml"
split_path = config_dir / "split_v1.yaml"

with protocol_path.open(encoding="utf-8") as handle:
    protocol = yaml.safe_load(handle)
with feature_path.open(encoding="utf-8") as handle:
    feature_contract = yaml.safe_load(handle)
with split_path.open(encoding="utf-8") as handle:
    split_contract = yaml.safe_load(handle)

cv_root = PROJECT_ROOT / protocol["contracts"]["cv_artifact"]
cv_manifest_path = cv_root / "manifest.json"
with cv_manifest_path.open(encoding="utf-8") as handle:
    cv_manifest = json.load(handle)

if protocol["dataset"] != "dataset_b":
    raise ValueError("Unexpected dataset in the model protocol.")
if protocol["protocol_version"] != "model_protocol_v1":
    raise ValueError("Unexpected model protocol version.")
if protocol["contracts"]["preprocessing_version"] != "dataset_b_preprocessing_v2":
    raise ValueError("Unexpected preprocessing contract.")
if feature_contract["feature_version"] != "temporary_training_lock_v0_2":
    raise ValueError("Unexpected feature lock.")
if feature_contract["feature_version"] != cv_manifest["feature_version"]:
    raise ValueError("Feature version differs between config and CV manifest.")
if split_contract["split_version"] != "split_v1":
    raise ValueError("Unexpected split version.")
if split_contract["split_version"] != cv_manifest["split_version"]:
    raise ValueError("Split version differs between config and CV manifest.")
if cv_manifest["preprocessing_version"] != "dataset_b_preprocessing_v2":
    raise ValueError("Unexpected preprocessing version.")
if (cv_manifest["source_partition"], cv_manifest["fit_scope"], cv_manifest["use"]) != (
    "train",
    "fold_train_only",
    "cross_validation_only",
):
    raise ValueError("CV preprocessing must be fold-local on Train.")
if cv_manifest["encoded_feature_count"] != 66:
    raise ValueError("Expected 66 encoded columns.")
if len(cv_manifest["output_columns"]) != 66:
    raise ValueError("CV manifest output column count differs from 66.")
if feature_contract["constraints"]["encoded_feature_count"] != 66:
    raise ValueError("Feature contract encoded column count differs from 66.")
if not set(cv_manifest["output_raw_sources"]) <= set(
    feature_contract["model_input_raw_columns"]
):
    raise ValueError("CV matrices use a forbidden raw input.")
if "xgb" not in cv_manifest["tree_profile_shared_by"]:
    raise ValueError("Tree profile is not declared for XGBoost.")
if cv_manifest["sample_weights_exported"] is not False:
    raise ValueError("Baseline CV must not use exported sample weights.")

forbidden_inputs = {
    *feature_contract["target"]["forbidden_model_inputs"],
    *feature_contract["audit_only_columns"],
}
if forbidden_inputs & set(feature_contract["model_input_raw_columns"]):
    raise ValueError("Target or audit columns appear in model inputs.")

expected_cv = {
    "splitter": "StratifiedKFold",
    "n_splits": 5,
    "shuffle": True,
    "random_state": 42,
}
for key, expected_value in expected_cv.items():
    if protocol["cross_validation"][key] != expected_value:
        raise ValueError(f"Model protocol CV mismatch: {key}.")
    if split_contract["cross_validation"][key] != expected_value:
        raise ValueError(f"Split contract CV mismatch: {key}.")
if protocol["cross_validation"]["population"] != "train_only":
    raise ValueError("Cross-validation population must be Train only.")
if split_contract["cross_validation"]["allowed_population"] != "train_only":
    raise ValueError("Split contract does not restrict CV to Train.")
if protocol["cross_validation"]["hyperparameter_search"] is not False:
    raise ValueError("Hyperparameter search is forbidden for this baseline.")
if protocol["cross_validation"]["trial_budget_per_model"] != 1:
    raise ValueError("XGB-01 allows exactly one fixed trial.")
if protocol["cross_validation"]["primary_metric"] != "roc_auc":
    raise ValueError("XGB-01 primary metric must be ROC-AUC.")
if any(protocol["class_handling"].values()):
    raise ValueError("Weighting or resampling is forbidden for this baseline.")
if protocol["runtime"]["early_stopping"] is not False:
    raise ValueError("Early stopping is forbidden for XGB-01.")
if protocol["runtime"]["eval_set"] is not False:
    raise ValueError("eval_set is forbidden for XGB-01.")

xgb_contract = protocol["models"]["xgb"]
xgb_parameters = dict(xgb_contract["parameters"])
expected_xgb_parameters = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.3,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "objective": "binary:logistic",
    "tree_method": "hist",
    "eval_metric": "logloss",
    "scale_pos_weight": 1.0,
    "random_state": 42,
}
if xgb_contract["estimator"] != "xgboost.XGBClassifier":
    raise ValueError("Unexpected XGBoost estimator.")
if xgb_contract["preprocessing_profile"] != "tree":
    raise ValueError("XGBoost must consume the shared tree profile.")
if xgb_parameters != expected_xgb_parameters:
    raise ValueError("XGBoost parameters differ from the fixed baseline contract.")

train = DatasetBSplitLoader().load("train")
train_ids = train.record_ids.to_numpy(dtype=str)
train_y = train.y.to_numpy(dtype=np.int8)
n_train = len(train_ids)

if n_train != 205_310:
    raise ValueError(f"Expected 205,310 Train rows; found {n_train:,}.")
if len(np.unique(train_ids)) != n_train:
    raise ValueError("Train record_id values are not unique.")
if not np.array_equal(np.unique(train_y), np.array([0, 1], dtype=np.int8)):
    raise ValueError("Expected both binary target classes in Train.")
if len(cv_manifest["folds"]) != 5:
    raise ValueError("Expected five folds in the CV manifest.")

fold_inputs = []
validation_counts = np.zeros(n_train, dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_number, ((expected_fit, expected_hold), fold) in enumerate(
    zip(
        splitter.split(np.zeros(n_train, dtype=np.uint8), train_y),
        cv_manifest["folds"],
    ),
    start=1,
):
    if fold["fold"] != fold_number:
        raise ValueError(f"Fold {fold_number}: manifest order differs.")
    if fold["directory"] != f"fold_{fold_number:02d}":
        raise ValueError(f"Fold {fold_number}: directory name differs.")

    fold_dir = cv_root / fold["directory"]
    rows_path = fold_dir / "rows.npz"
    x_fit_path = fold_dir / "tree_train.npz"
    x_hold_path = fold_dir / "tree_fold_validation.npz"

    for required_path in (rows_path, x_fit_path, x_hold_path):
        if not required_path.is_file():
            raise FileNotFoundError(required_path)

    with np.load(rows_path, allow_pickle=False) as rows:
        required_arrays = {
            "train_positions",
            "fold_validation_positions",
            "train_record_ids",
            "fold_validation_record_ids",
            "train_y",
            "fold_validation_y",
        }
        if set(rows.files) != required_arrays:
            raise ValueError(f"Fold {fold_number}: row metadata keys differ.")
        fit_positions = rows["train_positions"]
        hold_positions = rows["fold_validation_positions"]
        fit_ids = rows["train_record_ids"]
        hold_ids = rows["fold_validation_record_ids"]
        fit_y = rows["train_y"]
        hold_y = rows["fold_validation_y"]

    if not np.array_equal(fit_positions, expected_fit):
        raise ValueError(f"Fold {fold_number}: Train positions differ.")
    if not np.array_equal(hold_positions, expected_hold):
        raise ValueError(f"Fold {fold_number}: validation positions differ.")
    if not np.array_equal(train_ids[fit_positions], fit_ids):
        raise ValueError(f"Fold {fold_number}: Train record IDs differ.")
    if not np.array_equal(train_ids[hold_positions], hold_ids):
        raise ValueError(f"Fold {fold_number}: validation record IDs differ.")
    if not np.array_equal(train_y[fit_positions], fit_y):
        raise ValueError(f"Fold {fold_number}: Train labels differ.")
    if not np.array_equal(train_y[hold_positions], hold_y):
        raise ValueError(f"Fold {fold_number}: validation labels differ.")
    if len(fit_positions) != fold["train_rows"]:
        raise ValueError(f"Fold {fold_number}: Train row count differs.")
    if len(hold_positions) != fold["fold_validation_rows"]:
        raise ValueError(f"Fold {fold_number}: validation row count differs.")
    if len(np.unique(fit_y)) != 2 or len(np.unique(hold_y)) != 2:
        raise ValueError(f"Fold {fold_number}: a target class is missing.")

    x_fit = load_npz(x_fit_path)
    x_hold = load_npz(x_hold_path)
    if x_fit.shape != (len(fit_positions), 66):
        raise ValueError(f"Fold {fold_number}: Train matrix shape differs.")
    if x_hold.shape != (len(hold_positions), 66):
        raise ValueError(f"Fold {fold_number}: validation matrix shape differs.")
    if not np.isfinite(x_fit.data).all():
        raise ValueError(f"Fold {fold_number}: non-finite Train matrix value.")
    if not np.isfinite(x_hold.data).all():
        raise ValueError(f"Fold {fold_number}: non-finite validation matrix value.")
    del x_fit, x_hold

    validation_counts[hold_positions] += 1
    fold_inputs.append(
        {
            "fold": fold_number,
            "fit_positions": fit_positions,
            "hold_positions": hold_positions,
            "fit_y": fit_y,
            "hold_y": hold_y,
            "x_fit_path": x_fit_path,
            "x_hold_path": x_hold_path,
        }
    )
    print(
        f"Preflight fold {fold_number}: "
        f"{len(fit_y):,} train / {len(hold_y):,} validation, 66 columns"
    )

if not np.all(validation_counts == 1):
    raise ValueError("Every Train record must appear once as fold validation.")

source_hashes = {
    "model_protocol_v1.yaml": hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
    "feature_set_v1.yaml": hashlib.sha256(feature_path.read_bytes()).hexdigest(),
    "split_v1.yaml": hashlib.sha256(split_path.read_bytes()).hexdigest(),
    "cv_preprocessing_manifest.json": hashlib.sha256(
        cv_manifest_path.read_bytes()
    ).hexdigest(),
}

artifact_dir = PROJECT_ROOT / "artifacts/dataset_b/xgb_baseline_v1"
results_path = artifact_dir / "cv_results.json"
oof_path = artifact_dir / "cv_oof_predictions.csv"

print(f"Preflight complete: {n_train:,} Train rows and five valid folds.")
print(f"Fixed XGBoost parameters: {xgb_parameters}")
print("Runtime library versions:")
display(pd.Series(runtime_versions, name="runtime"))
print("Preprocessing artifact library versions:")
display(pd.Series(cv_manifest["library_versions"], name="preprocessing_artifact"))


Preflight fold 1: 164,248 train / 41,062 validation, 66 columns
Preflight fold 2: 164,248 train / 41,062 validation, 66 columns
Preflight fold 3: 164,248 train / 41,062 validation, 66 columns
Preflight fold 4: 164,248 train / 41,062 validation, 66 columns
Preflight fold 5: 164,248 train / 41,062 validation, 66 columns
Preflight complete: 205,310 Train rows and five valid folds.
Fixed XGBoost parameters: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3, 'subsample': 1.0, 'colsample_bytree': 1.0, 'objective': 'binary:logistic', 'tree_method': 'hist', 'eval_metric': 'logloss', 'scale_pos_weight': 1.0, 'random_state': 42}
Runtime library versions:


python          3.12.13
numpy             2.5.2
pandas            3.0.5
scipy            1.18.1
scikit-learn      1.9.1
xgboost           3.2.0
Name: runtime, dtype: str

Preprocessing artifact library versions:


numpy            2.3.5
pandas           2.3.3
scikit-learn     1.7.2
scipy           1.16.3
xgboost          3.2.0
Name: preprocessing_artifact, dtype: str

## 2. Fit năm fold và lưu OOF artifact\n
\n
Mỗi fold tạo một `XGBClassifier` mới từ tham số trong protocol. Artifact chỉ được ghi sau khi cả năm fold hoàn tất. Nếu hai artifact đã tồn tại, cell chỉ kiểm tra lại rồi bỏ qua fit.\n

In [3]:
import time
import warnings

if results_path.exists() != oof_path.exists():
    raise FileExistsError(
        "Only one XGBoost CV artifact exists; inspect before retrying."
    )

if results_path.exists():
    with results_path.open(encoding="utf-8") as handle:
        cv_results = json.load(handle)
    cv_oof_predictions = pd.read_csv(oof_path, dtype={"record_id": str})

    expected_folds = np.zeros(n_train, dtype=np.int8)
    for fold in fold_inputs:
        expected_folds[fold["hold_positions"]] = fold["fold"]

    stored_versions = cv_results.get("library_versions", {})
    if (
        cv_results.get("model") != "xgb"
        or cv_results.get("status") not in ("complete", "complete_with_warnings")
        or cv_results.get("protocol_version") != protocol["protocol_version"]
        or cv_results.get("feature_version") != feature_contract["feature_version"]
        or cv_results.get("split_version") != split_contract["split_version"]
        or cv_results.get("preprocessing_version")
        != cv_manifest["preprocessing_version"]
        or cv_results.get("parameters") != xgb_parameters
        or cv_results.get("source_sha256") != source_hashes
        or cv_results.get("fold_count") != 5
        or cv_results.get("train_rows") != n_train
        or len(cv_results.get("folds", [])) != 5
        or stored_versions.get("xgboost") != "3.2.0"
        or list(cv_oof_predictions.columns)
        != ["record_id", "y_approved", "fold", "p_approved"]
        or len(cv_oof_predictions) != n_train
        or not np.array_equal(
            cv_oof_predictions["record_id"].to_numpy(dtype=str), train_ids
        )
        or not np.array_equal(
            cv_oof_predictions["y_approved"].to_numpy(dtype=np.int8), train_y
        )
        or not np.array_equal(
            cv_oof_predictions["fold"].to_numpy(dtype=np.int8), expected_folds
        )
    ):
        raise ValueError(
            "Existing XGBoost CV artifacts differ from the locked Train contract."
        )

    probabilities = cv_oof_predictions["p_approved"].to_numpy(dtype=float)
    if not np.isfinite(probabilities).all():
        raise ValueError("Existing OOF probabilities contain non-finite values.")
    if np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("Existing OOF probabilities are outside [0, 1].")

    auc_values = []
    for fold, result in zip(fold_inputs, cv_results["folds"]):
        hold_positions = fold["hold_positions"]
        auc = float(
            roc_auc_score(train_y[hold_positions], probabilities[hold_positions])
        )
        if result["fold"] != fold["fold"]:
            raise ValueError(f"Fold {fold['fold']}: stored fold number differs.")
        if not np.isclose(auc, result["roc_auc"], rtol=0, atol=1e-12):
            raise ValueError(f"Fold {fold['fold']}: stored ROC-AUC differs.")
        auc_values.append(auc)

    if not np.isclose(
        np.mean(auc_values),
        cv_results["mean_roc_auc"],
        rtol=0,
        atol=1e-12,
    ):
        raise ValueError("Stored mean ROC-AUC differs from OOF probabilities.")
    if not np.isclose(
        np.std(auc_values, ddof=0),
        cv_results["std_roc_auc"],
        rtol=0,
        atol=1e-12,
    ):
        raise ValueError("Stored ROC-AUC standard deviation differs.")
    if cv_results.get("std_ddof") != 0:
        raise ValueError("Stored ROC-AUC standard deviation must use ddof=0.")

    print("Existing XGBoost CV artifacts verified; model fit skipped.")
else:
    oof_probability = np.full(n_train, np.nan, dtype=float)
    oof_fold = np.zeros(n_train, dtype=np.int8)
    fold_results = []

    for fold in fold_inputs:
        fold_number = fold["fold"]
        x_fit = load_npz(fold["x_fit_path"])
        x_hold = load_npz(fold["x_hold_path"])
        model = XGBClassifier(**xgb_parameters)

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            fit_start = time.perf_counter()
            model.fit(x_fit, fold["fit_y"])
            fit_seconds = time.perf_counter() - fit_start

            if not np.array_equal(model.classes_, np.array([0, 1])):
                raise ValueError(
                    f"Fold {fold_number}: unexpected fitted class order."
                )

            predict_start = time.perf_counter()
            probability = model.predict_proba(x_hold)[:, 1]
            predict_seconds = time.perf_counter() - predict_start

        if len(probability) != len(fold["hold_y"]):
            raise ValueError(f"Fold {fold_number}: probability length differs.")
        if not np.isfinite(probability).all():
            raise ValueError(f"Fold {fold_number}: non-finite probability.")
        if np.any((probability < 0) | (probability > 1)):
            raise ValueError(f"Fold {fold_number}: probability outside [0, 1].")

        hold_positions = fold["hold_positions"]
        if np.any(oof_fold[hold_positions] != 0):
            raise ValueError(f"Fold {fold_number}: duplicate OOF position.")

        oof_probability[hold_positions] = probability
        oof_fold[hold_positions] = fold_number
        auc = float(roc_auc_score(fold["hold_y"], probability))
        warning_messages = [
            f"{warning.category.__name__}: {warning.message}"
            for warning in caught
        ]
        fold_results.append(
            {
                "fold": fold_number,
                "train_rows": len(fold["fit_y"]),
                "fold_validation_rows": len(fold["hold_y"]),
                "roc_auc": auc,
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds,
                "warnings": warning_messages,
            }
        )

        print(
            f"Fold {fold_number}: ROC-AUC={auc:.6f}, "
            f"fit={fit_seconds:.2f}s, predict={predict_seconds:.2f}s, "
            f"warnings={len(warning_messages)}"
        )
        for message in warning_messages:
            print(f"  WARNING: {message}")

    if not np.all(oof_fold > 0):
        raise ValueError("At least one Train row has no OOF fold.")
    if not np.isfinite(oof_probability).all():
        raise ValueError("At least one Train row has no finite OOF probability.")

    cv_oof_predictions = pd.DataFrame(
        {
            "record_id": train_ids,
            "y_approved": train_y,
            "fold": oof_fold,
            "p_approved": oof_probability,
        }
    )
    auc_values = np.asarray(
        [result["roc_auc"] for result in fold_results],
        dtype=float,
    )
    cv_results = {
        "status": (
            "complete_with_warnings"
            if any(result["warnings"] for result in fold_results)
            else "complete"
        ),
        "model": "xgb",
        "estimator": xgb_contract["estimator"],
        "protocol_version": protocol["protocol_version"],
        "feature_version": feature_contract["feature_version"],
        "split_version": split_contract["split_version"],
        "preprocessing_version": cv_manifest["preprocessing_version"],
        "population": "train_only",
        "fold_count": 5,
        "train_rows": n_train,
        "mean_roc_auc": float(auc_values.mean()),
        "std_roc_auc": float(auc_values.std(ddof=0)),
        "std_ddof": 0,
        "parameters": xgb_parameters,
        "source_sha256": source_hashes,
        "library_versions": runtime_versions,
        "preprocessing_library_versions": cv_manifest["library_versions"],
        "folds": fold_results,
    }

    artifact_dir.mkdir(parents=True, exist_ok=True)
    cv_oof_predictions.to_csv(
        oof_path,
        index=False,
        float_format="%.17g",
    )
    with results_path.open("w", encoding="utf-8") as handle:
        json.dump(cv_results, handle, ensure_ascii=False, indent=2)
        handle.write("\n")

    print(f"Saved {oof_path}")
    print(f"Saved {results_path}")

print(f"CV status: {cv_results['status']}")
print(f"Mean ROC-AUC: {cv_results['mean_roc_auc']:.6f}")
print(f"Std ROC-AUC (ddof=0): {cv_results['std_roc_auc']:.6f}")
display(pd.DataFrame(cv_results["folds"]).drop(columns=["warnings"]))
display(cv_oof_predictions.head())


Fold 1: ROC-AUC=0.860227, fit=0.53s, predict=0.02s, warnings=0
Fold 2: ROC-AUC=0.858505, fit=0.42s, predict=0.02s, warnings=0
Fold 3: ROC-AUC=0.859707, fit=0.43s, predict=0.02s, warnings=0
Fold 4: ROC-AUC=0.857115, fit=0.39s, predict=0.03s, warnings=0
Fold 5: ROC-AUC=0.861802, fit=0.38s, predict=0.02s, warnings=0
Saved d:\FPTU_Study\FA2026\DAP391m_GR12\artifacts\dataset_b\xgb_baseline_v1\cv_oof_predictions.csv
Saved d:\FPTU_Study\FA2026\DAP391m_GR12\artifacts\dataset_b\xgb_baseline_v1\cv_results.json
CV status: complete
Mean ROC-AUC: 0.859471
Std ROC-AUC (ddof=0): 0.001584


,fold,train_rows,fold_validation_rows,roc_auc,fit_seconds,predict_seconds
0,1,164248,41062,0.860227,0.534496,0.016879
1,2,164248,41062,0.858505,0.421662,0.022225
2,3,164248,41062,0.859707,0.427600,0.017935
3,4,164248,41062,0.857115,0.385096,0.025897
4,5,164248,41062,0.861802,0.378847,0.017451


,record_id,y_approved,fold,p_approved
0,dataset_b-000343629,1,5,0.937527
1,dataset_b-000256425,1,4,0.945475
2,dataset_b-000342696,1,2,0.967328
3,dataset_b-000235315,0,2,0.245035
4,dataset_b-000271985,1,5,0.933919


## XGB-01 result boundary\n
\n
Kết quả của notebook này dừng tại Train-only five-fold CV và OOF predictions. Full-Train refit, Validation threshold selection và Test evaluation thuộc XGB-02, XGB-03 và XGB-04; không được thực hiện trong XGB-01.\n

## XGB-02 — Refit toàn Train và dự báo Validation

Tạo một pipeline mới gồm preprocessing profile `xgb` và estimator cố định. Pipeline chỉ fit trên toàn Train; Validation chỉ được dùng để tạo `P(Approved)`. Không chọn threshold, không tính Validation metric và không đọc Test trong XGB-02.


In [4]:
import joblib
from sklearn.pipeline import Pipeline
from src.dataset_b.preprocessing import DatasetBSplitLoader, make_preprocessing_pipeline

xgb02_artifact_dir = PROJECT_ROOT / "artifacts/dataset_b/xgb_baseline_v1"
xgb02_cv_results_path = xgb02_artifact_dir / "cv_results.json"
xgb02_cv_oof_path = xgb02_artifact_dir / "cv_oof_predictions.csv"
xgb02_split_manifest_path = PROJECT_ROOT / "artifacts/dataset_b/split_v1.json"

for required_path in (
    xgb02_cv_results_path,
    xgb02_cv_oof_path,
    xgb02_split_manifest_path,
):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

with xgb02_cv_results_path.open(encoding="utf-8") as handle:
    xgb02_cv_results = json.load(handle)
with xgb02_split_manifest_path.open(encoding="utf-8") as handle:
    xgb02_split_manifest = json.load(handle)

if xgb02_cv_results["status"] not in ("complete", "complete_with_warnings"):
    raise ValueError("XGB-01 must be complete before XGB-02.")
if xgb02_cv_results["model"] != "xgb":
    raise ValueError("XGB-01 artifact has an unexpected model.")
if xgb02_cv_results["parameters"] != xgb_parameters:
    raise ValueError("XGB-01 parameters differ from the locked XGBoost contract.")
if xgb02_cv_results["source_sha256"] != source_hashes:
    raise ValueError("XGB-01 source hashes differ from the current contract.")
if xgb02_cv_results["train_rows"] != 205_310:
    raise ValueError("XGB-01 Train row count differs from the fixed split.")

if protocol["refit"]["population"] != "full_train":
    raise ValueError("XGB-02 must refit on full Train.")
if protocol["refit"]["reinitialize_preprocessing"] is not True:
    raise ValueError("XGB-02 requires a new preprocessing pipeline.")
if protocol["refit"]["reinitialize_model"] is not True:
    raise ValueError("XGB-02 requires a new estimator.")
if protocol["validation"]["use"] != "threshold_selection":
    raise ValueError("Validation has an unexpected protocol role.")
if protocol["validation"]["may_refit_preprocessing_or_model"] is not False:
    raise ValueError("Validation must not refit preprocessing or the estimator.")

xgb02_loader = DatasetBSplitLoader()
xgb02_full_train = xgb02_loader.load("train")
xgb02_validation = xgb02_loader.load("validation")
xgb02_expected_features = feature_contract["model_input_raw_columns"]

if list(xgb02_full_train.X.columns) != xgb02_expected_features:
    raise ValueError("Full-Train raw feature order differs from the contract.")
if list(xgb02_validation.X.columns) != xgb02_expected_features:
    raise ValueError("Validation raw feature order differs from the contract.")
if len(xgb02_full_train.X) != 205_310:
    raise ValueError("Unexpected full-Train row count.")
if len(xgb02_validation.X) != 43_995:
    raise ValueError("Unexpected Validation row count.")
if xgb02_full_train.record_ids.tolist() != xgb02_split_manifest["train_record_ids"]:
    raise ValueError("Full-Train record order differs from the split manifest.")
if xgb02_validation.record_ids.tolist() != xgb02_split_manifest["validation_record_ids"]:
    raise ValueError("Validation record order differs from the split manifest.")
if not xgb02_full_train.record_ids.is_unique:
    raise ValueError("Full-Train record IDs are not unique.")
if not xgb02_validation.record_ids.is_unique:
    raise ValueError("Validation record IDs are not unique.")
if np.intersect1d(
    xgb02_full_train.record_ids.to_numpy(),
    xgb02_validation.record_ids.to_numpy(),
).size:
    raise ValueError("Train and Validation record IDs overlap.")
if not np.array_equal(xgb02_full_train.record_ids.to_numpy(dtype=str), train_ids):
    raise ValueError("XGB-02 full-Train order differs from the XGB-01 Train order.")

xgb02_forbidden_inputs = {
    "record_id",
    "action_taken",
    "y_approved",
    *feature_contract["audit_only_columns"],
}
if xgb02_forbidden_inputs & set(xgb02_expected_features):
    raise ValueError("Identifier, target, or audit data appears in X.")
if set(xgb02_full_train.X.columns) & set(xgb02_full_train.audit.columns):
    raise ValueError("Full-Train audit columns appear in X.")
if set(xgb02_validation.X.columns) & set(xgb02_validation.audit.columns):
    raise ValueError("Validation audit columns appear in X.")

xgb02_pipeline = Pipeline(
    [
        ("preprocess", make_preprocessing_pipeline("xgb")),
        ("estimator", XGBClassifier(**xgb_parameters)),
    ]
)

with warnings.catch_warnings(record=True) as xgb02_caught_warnings:
    warnings.simplefilter("always")
    xgb02_fit_start = time.perf_counter()
    xgb02_pipeline.fit(xgb02_full_train.X, xgb02_full_train.y)
    xgb02_fit_seconds = time.perf_counter() - xgb02_fit_start

xgb02_warning_messages = [
    f"{item.category.__name__}: {item.message}"
    for item in xgb02_caught_warnings
]
xgb02_estimator = xgb02_pipeline.named_steps["estimator"]
xgb02_preprocessor = xgb02_pipeline.named_steps["preprocess"]

if not np.array_equal(xgb02_estimator.classes_, np.array([0, 1])):
    raise ValueError("Full-Train XGBoost class order differs from [0, 1].")
if int(xgb02_estimator.n_features_in_) != 66:
    raise ValueError("Full-Train XGBoost estimator did not receive 66 columns.")
if len(xgb02_preprocessor.get_feature_names_out()) != 66:
    raise ValueError("Full-Train preprocessing output does not contain 66 columns.")

xgb02_predict_start = time.perf_counter()
xgb02_validation_probability = xgb02_pipeline.predict_proba(
    xgb02_validation.X
)[:, 1]
xgb02_validation_predict_seconds = (
    time.perf_counter() - xgb02_predict_start
)

if len(xgb02_validation_probability) != len(xgb02_validation.X):
    raise ValueError("Validation probability length differs from Validation rows.")
if not np.isfinite(xgb02_validation_probability).all():
    raise ValueError("Validation probabilities contain non-finite values.")
if np.any(
    (xgb02_validation_probability < 0)
    | (xgb02_validation_probability > 1)
):
    raise ValueError("Validation probabilities are outside [0, 1].")

xgb02_validation_predictions = pd.DataFrame(
    {
        "record_id": xgb02_validation.record_ids.to_numpy(dtype=str),
        "y_approved": xgb02_validation.y.to_numpy(dtype=np.int8),
        "p_approved": xgb02_validation_probability,
    }
)
if len(xgb02_validation_predictions) != 43_995:
    raise ValueError("Validation prediction row count differs from 43,995.")
if not xgb02_validation_predictions["record_id"].is_unique:
    raise ValueError("Validation prediction record IDs are not unique.")

print(
    f"Full-Train XGBoost fitted on {len(xgb02_full_train.X):,} rows "
    f"in {xgb02_fit_seconds:.2f}s."
)
print(
    f"Validation probabilities generated for "
    f"{len(xgb02_validation_predictions):,} rows "
    f"in {xgb02_validation_predict_seconds:.2f}s."
)
print(f"Fit warnings: {len(xgb02_warning_messages)}")
for message in xgb02_warning_messages:
    print(f"  WARNING: {message}")


Dataset B LTV >1000% retained for 42 rows
Dataset B LTV >1000% retained for 7 rows


Full-Train XGBoost fitted on 205,310 rows in 3.18s.
Validation probabilities generated for 43,995 rows in 0.48s.
Fit warnings: 0


## XGB-02 — Lưu và reload fitted pipeline

Ba artifact XGB-02 được kiểm tra theo bộ: model, Validation predictions và run manifest. Hai artifact XGB-01 được giữ nguyên và liên kết bằng SHA-256 trong run manifest.


In [5]:
xgb02_model_path = xgb02_artifact_dir / "xgb_full_train.joblib"
xgb02_prediction_path = xgb02_artifact_dir / "validation_predictions.csv"
xgb02_manifest_path = xgb02_artifact_dir / "run_manifest.json"
xgb02_artifact_paths = (
    xgb02_model_path,
    xgb02_prediction_path,
    xgb02_manifest_path,
)

xgb02_cv_lineage = {
    "cv_results": {
        "file": xgb02_cv_results_path.name,
        "sha256": hashlib.sha256(
            xgb02_cv_results_path.read_bytes()
        ).hexdigest(),
    },
    "cv_oof_predictions": {
        "file": xgb02_cv_oof_path.name,
        "sha256": hashlib.sha256(
            xgb02_cv_oof_path.read_bytes()
        ).hexdigest(),
    },
}

xgb02_run_manifest = {
    "schema_version": 1,
    "dataset": "dataset_b",
    "run_id": "xgb_baseline_v1",
    "stage": "full_train_refit_validation_predictions",
    "model": "xgb",
    "model_protocol_version": protocol["protocol_version"],
    "feature_version": feature_contract["feature_version"],
    "split_version": split_contract["split_version"],
    "preprocessing_version": cv_manifest["preprocessing_version"],
    "fit_partition": "train",
    "fit_rows": len(xgb02_full_train.X),
    "prediction_partition": "validation",
    "prediction_rows": len(xgb02_validation_predictions),
    "raw_feature_columns": xgb02_expected_features,
    "encoded_feature_count": 66,
    "model_parameters": xgb_parameters,
    "class_order": [0, 1],
    "probability_column": "p_approved",
    "fit_seconds": xgb02_fit_seconds,
    "validation_predict_seconds": xgb02_validation_predict_seconds,
    "fit_warnings": xgb02_warning_messages,
    "library_versions": {
        **runtime_versions,
        "joblib": joblib.__version__,
    },
    "xgb01_cv_lineage": xgb02_cv_lineage,
    "artifacts": {
        "fitted_pipeline": xgb02_model_path.name,
        "validation_predictions": xgb02_prediction_path.name,
        "run_manifest": xgb02_manifest_path.name,
    },
}

xgb02_core_manifest_keys = (
    "schema_version",
    "dataset",
    "run_id",
    "stage",
    "model",
    "model_protocol_version",
    "feature_version",
    "split_version",
    "preprocessing_version",
    "fit_partition",
    "fit_rows",
    "prediction_partition",
    "prediction_rows",
    "raw_feature_columns",
    "encoded_feature_count",
    "model_parameters",
    "class_order",
    "probability_column",
    "library_versions",
    "xgb01_cv_lineage",
    "artifacts",
)

if any(path.exists() for path in xgb02_artifact_paths):
    if not all(path.is_file() for path in xgb02_artifact_paths):
        raise FileExistsError(
            "Partial XGB-02 artifacts exist; refusing to overwrite them."
        )
    with xgb02_manifest_path.open(encoding="utf-8") as handle:
        xgb02_existing_manifest = json.load(handle)
    for key in xgb02_core_manifest_keys:
        if xgb02_existing_manifest.get(key) != xgb02_run_manifest[key]:
            raise FileExistsError(
                f"Existing XGB-02 artifact has a different {key}; "
                "refusing to overwrite."
            )
    xgb02_expected_artifact_hashes = {
        "fitted_pipeline_sha256": hashlib.sha256(
            xgb02_model_path.read_bytes()
        ).hexdigest(),
        "validation_predictions_sha256": hashlib.sha256(
            xgb02_prediction_path.read_bytes()
        ).hexdigest(),
    }
    if (
        xgb02_existing_manifest.get("artifact_sha256")
        != xgb02_expected_artifact_hashes
    ):
        raise FileExistsError(
            "Existing XGB-02 artifact hashes differ from the run manifest."
        )
    print("Existing XGB-02 artifacts found; verifying without overwriting.")
else:
    xgb02_artifact_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(xgb02_pipeline, xgb02_model_path)
    xgb02_validation_predictions.to_csv(
        xgb02_prediction_path,
        index=False,
        float_format="%.17g",
    )
    xgb02_run_manifest["artifact_sha256"] = {
        "fitted_pipeline_sha256": hashlib.sha256(
            xgb02_model_path.read_bytes()
        ).hexdigest(),
        "validation_predictions_sha256": hashlib.sha256(
            xgb02_prediction_path.read_bytes()
        ).hexdigest(),
    }
    with xgb02_manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(
            xgb02_run_manifest,
            handle,
            ensure_ascii=False,
            indent=2,
        )
        handle.write("\n")
    print(
        "Created XGBoost full-Train model, Validation predictions, "
        "and run manifest."
    )

xgb02_loaded_pipeline = joblib.load(xgb02_model_path)
xgb02_saved_predictions = pd.read_csv(
    xgb02_prediction_path,
    dtype={"record_id": str},
)
with xgb02_manifest_path.open(encoding="utf-8") as handle:
    xgb02_saved_manifest = json.load(handle)

if list(xgb02_saved_predictions.columns) != [
    "record_id",
    "y_approved",
    "p_approved",
]:
    raise ValueError("Saved Validation prediction columns differ.")
if len(xgb02_saved_predictions) != len(xgb02_validation_predictions):
    raise ValueError("Saved Validation prediction row count differs.")
if not xgb02_saved_predictions["record_id"].is_unique:
    raise ValueError("Saved Validation record IDs are not unique.")
if not np.array_equal(
    xgb02_saved_predictions["record_id"].to_numpy(dtype=str),
    xgb02_validation.record_ids.to_numpy(dtype=str),
):
    raise ValueError("Saved Validation record order differs.")
if not np.array_equal(
    xgb02_saved_predictions["y_approved"].to_numpy(dtype=np.int8),
    xgb02_validation.y.to_numpy(dtype=np.int8),
):
    raise ValueError("Saved Validation labels differ.")
if not np.allclose(
    xgb02_saved_predictions["p_approved"].to_numpy(dtype=float),
    xgb02_validation_probability,
    rtol=0,
    atol=1e-15,
):
    raise ValueError("Saved Validation probabilities differ.")

xgb02_reloaded_probability = xgb02_loaded_pipeline.predict_proba(
    xgb02_validation.X
)[:, 1]
if not np.allclose(
    xgb02_reloaded_probability,
    xgb02_validation_probability,
    rtol=0,
    atol=1e-12,
):
    raise ValueError("Reloaded pipeline probabilities differ.")

xgb02_loaded_estimator = xgb02_loaded_pipeline.named_steps["estimator"]
if not np.array_equal(
    xgb02_loaded_estimator.classes_,
    np.array([0, 1]),
):
    raise ValueError("Reloaded estimator class order differs.")
if int(xgb02_loaded_estimator.n_features_in_) != 66:
    raise ValueError("Reloaded estimator feature count differs.")
for key, expected_value in xgb_parameters.items():
    if xgb02_loaded_estimator.get_params()[key] != expected_value:
        raise ValueError(
            f"Reloaded XGBoost parameter differs: {key}."
        )

for key in xgb02_core_manifest_keys:
    if xgb02_saved_manifest.get(key) != xgb02_run_manifest[key]:
        raise ValueError(f"Saved run manifest differs: {key}.")
xgb02_saved_hashes = xgb02_saved_manifest.get("artifact_sha256", {})
if xgb02_saved_hashes.get(
    "fitted_pipeline_sha256"
) != hashlib.sha256(xgb02_model_path.read_bytes()).hexdigest():
    raise ValueError("Saved model hash differs from the run manifest.")
if xgb02_saved_hashes.get(
    "validation_predictions_sha256"
) != hashlib.sha256(xgb02_prediction_path.read_bytes()).hexdigest():
    raise ValueError(
        "Saved Validation prediction hash differs from the run manifest."
    )

print(
    f"Reload validation passed: "
    f"{len(xgb02_saved_predictions):,} ordered record IDs "
    "and matching probabilities."
)
display(xgb02_saved_predictions.head())


Created XGBoost full-Train model, Validation predictions, and run manifest.


Dataset B LTV >1000% retained for 7 rows


Reload validation passed: 43,995 ordered record IDs and matching probabilities.


,record_id,y_approved,p_approved
0,dataset_b-000299096,1,0.938569
1,dataset_b-000172161,0,0.294190
2,dataset_b-000065193,0,0.103035
3,dataset_b-000291201,0,0.258560
4,dataset_b-000137115,1,0.724743


## XGB-02 result boundary

XGB-02 kết thúc tại fitted full-Train pipeline và xác suất Validation theo đúng thứ tự `record_id` của manifest. Threshold selection thuộc XGB-03; Test evaluation thuộc XGB-04.


## XGB-03 — Chọn và khóa threshold trên Validation

Dùng duy nhất xác suất Validation đã lưu bởi XGB-02. Xét threshold từ 0,001 đến 0,999 với bước 0,001; tối đa hóa Balanced Accuracy; nếu hòa, chọn threshold gần 0,5 nhất rồi chọn threshold thấp hơn. Không refit model và không đọc Test.


In [8]:
import hashlib
import json

import numpy as np
import pandas as pd

xgb03_artifact_dir = PROJECT_ROOT / "artifacts/dataset_b/xgb_baseline_v1"
xgb03_prediction_path = xgb03_artifact_dir / "validation_predictions.csv"
xgb03_manifest_path = xgb03_artifact_dir / "run_manifest.json"
xgb03_model_path = xgb03_artifact_dir / "xgb_full_train.joblib"
xgb03_validation_path = (
    PROJECT_ROOT / "data/dataset_b/split_v1/validation.csv"
)
xgb03_split_manifest_path = (
    PROJECT_ROOT / "artifacts/dataset_b/split_v1.json"
)
xgb03_candidates_path = xgb03_artifact_dir / "threshold_candidates.csv"
xgb03_selection_path = xgb03_artifact_dir / "threshold_selection.json"

for required_path in (
    xgb03_prediction_path,
    xgb03_manifest_path,
    xgb03_model_path,
    xgb03_validation_path,
    xgb03_split_manifest_path,
):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

with xgb03_manifest_path.open(encoding="utf-8") as handle:
    xgb03_manifest = json.load(handle)
with xgb03_split_manifest_path.open(encoding="utf-8") as handle:
    xgb03_split_manifest = json.load(handle)

if xgb03_manifest["run_id"] != "xgb_baseline_v1":
    raise ValueError("Unexpected XGBoost run ID.")
if xgb03_manifest["stage"] != "full_train_refit_validation_predictions":
    raise ValueError("XGB-02 stage is not complete.")
if xgb03_manifest["model"] != "xgb":
    raise ValueError("Unexpected model in the XGB-02 run manifest.")
if xgb03_manifest["prediction_partition"] != "validation":
    raise ValueError("XGB-03 requires Validation predictions.")
if xgb03_manifest["prediction_rows"] != 43_995:
    raise ValueError("Unexpected Validation prediction row count.")
if xgb03_manifest["feature_version"] != feature_contract["feature_version"]:
    raise ValueError("Feature version differs from the current contract.")
if xgb03_manifest["split_version"] != split_contract["split_version"]:
    raise ValueError("Split version differs from the current contract.")
if (
    xgb03_manifest["preprocessing_version"]
    != cv_manifest["preprocessing_version"]
):
    raise ValueError("Preprocessing version differs from the current contract.")
if (
    xgb03_manifest["model_protocol_version"]
    != protocol["protocol_version"]
):
    raise ValueError("Model protocol version differs.")
if xgb03_manifest["model_parameters"] != xgb_parameters:
    raise ValueError("XGB-02 parameters differ from the locked baseline.")
if xgb03_manifest["probability_column"] != "p_approved":
    raise ValueError("Unexpected Validation probability column.")
if protocol["validation"]["use"] != "threshold_selection":
    raise ValueError("Validation is not assigned to threshold selection.")
if protocol["validation"]["objective"] != "balanced_accuracy":
    raise ValueError("Unexpected Validation threshold objective.")
if protocol["validation"]["may_refit_preprocessing_or_model"] is not False:
    raise ValueError("Validation must not refit preprocessing or the model.")

xgb03_prediction_hash = hashlib.sha256(
    xgb03_prediction_path.read_bytes()
).hexdigest()
xgb03_model_hash = hashlib.sha256(
    xgb03_model_path.read_bytes()
).hexdigest()
if (
    xgb03_prediction_hash
    != xgb03_manifest["artifact_sha256"][
        "validation_predictions_sha256"
    ]
):
    raise ValueError("Validation prediction hash differs from run manifest.")
if (
    xgb03_model_hash
    != xgb03_manifest["artifact_sha256"]["fitted_pipeline_sha256"]
):
    raise ValueError("Fitted model hash differs from run manifest.")

xgb03_predictions = pd.read_csv(
    xgb03_prediction_path,
    dtype={"record_id": str},
)
xgb03_validation = pd.read_csv(
    xgb03_validation_path,
    usecols=["record_id", "y_approved"],
    dtype={"record_id": str},
)

if list(xgb03_predictions.columns) != [
    "record_id",
    "y_approved",
    "p_approved",
]:
    raise ValueError("Unexpected Validation prediction columns.")
if len(xgb03_predictions) != len(xgb03_validation):
    raise ValueError("Validation prediction and source row counts differ.")
if len(xgb03_predictions) != xgb03_manifest["prediction_rows"]:
    raise ValueError("Validation prediction count differs from run manifest.")
if not xgb03_predictions["record_id"].is_unique:
    raise ValueError("Validation prediction record IDs are not unique.")
if (
    xgb03_predictions["record_id"].tolist()
    != xgb03_split_manifest["validation_record_ids"]
):
    raise ValueError("Validation prediction order differs from split manifest.")
if not np.array_equal(
    xgb03_predictions["record_id"].to_numpy(dtype=str),
    xgb03_validation["record_id"].to_numpy(dtype=str),
):
    raise ValueError("Validation prediction IDs differ from source rows.")
if not np.array_equal(
    xgb03_predictions["y_approved"].to_numpy(dtype=np.int8),
    xgb03_validation["y_approved"].to_numpy(dtype=np.int8),
):
    raise ValueError("Validation prediction labels differ from source rows.")

xgb03_y = xgb03_predictions["y_approved"].to_numpy(dtype=np.int8)
xgb03_p = xgb03_predictions["p_approved"].to_numpy(dtype=float)
if set(np.unique(xgb03_y)) != {0, 1}:
    raise ValueError("Validation must contain both target classes.")
if not np.isfinite(xgb03_p).all():
    raise ValueError("Validation probabilities contain non-finite values.")
if np.any((xgb03_p < 0) | (xgb03_p > 1)):
    raise ValueError("Validation probabilities are outside [0, 1].")

xgb03_positive = xgb03_y == 1
xgb03_negative = xgb03_y == 0
xgb03_n_approved = int(xgb03_positive.sum())
xgb03_n_denied = int(xgb03_negative.sum())
if xgb03_n_approved == 0 or xgb03_n_denied == 0:
    raise ValueError("Balanced Accuracy requires both Validation classes.")

xgb03_grid = np.arange(1, 1000, dtype=np.int64)
xgb03_thresholds = xgb03_grid / 1000.0
xgb03_tp = np.asarray(
    [
        np.count_nonzero(xgb03_p[xgb03_positive] >= threshold)
        for threshold in xgb03_thresholds
    ],
    dtype=np.int64,
)
xgb03_tn = np.asarray(
    [
        np.count_nonzero(xgb03_p[xgb03_negative] < threshold)
        for threshold in xgb03_thresholds
    ],
    dtype=np.int64,
)
xgb03_fn = xgb03_n_approved - xgb03_tp
xgb03_fp = xgb03_n_denied - xgb03_tn

# Fixed class denominators make this integer score rank
# Balanced Accuracy exactly without floating-point tie ambiguity.
xgb03_score = (
    xgb03_tp * xgb03_n_denied
    + xgb03_tn * xgb03_n_approved
)
xgb03_max_indices = np.flatnonzero(
    xgb03_score == xgb03_score.max()
)
xgb03_selected_index = min(
    xgb03_max_indices,
    key=lambda index: (
        abs(int(xgb03_grid[index]) - 500),
        int(xgb03_grid[index]),
    ),
)
xgb03_balanced_accuracy = xgb03_score / (
    2 * xgb03_n_approved * xgb03_n_denied
)

xgb03_candidates = pd.DataFrame(
    {
        "threshold": xgb03_thresholds,
        "tp_approved": xgb03_tp,
        "fn_approved": xgb03_fn,
        "tn_denied": xgb03_tn,
        "fp_denied": xgb03_fp,
        "n_approved": xgb03_n_approved,
        "n_denied": xgb03_n_denied,
        "recall_approved": xgb03_tp / xgb03_n_approved,
        "recall_denied": xgb03_tn / xgb03_n_denied,
        "balanced_accuracy": xgb03_balanced_accuracy,
        "is_maximum": xgb03_score == xgb03_score.max(),
        "is_selected": (
            np.arange(len(xgb03_grid)) == xgb03_selected_index
        ),
    }
)
if len(xgb03_candidates) != 999:
    raise ValueError("Expected 999 threshold candidates.")
if int(xgb03_candidates["is_selected"].sum()) != 1:
    raise ValueError("Exactly one threshold must be selected.")

xgb03_candidates_csv = xgb03_candidates.to_csv(
    index=False,
    float_format="%.17g",
)
xgb03_validation_id_hash = hashlib.sha256(
    (
        "\n".join(xgb03_predictions["record_id"].tolist())
        + "\n"
    ).encode("utf-8")
).hexdigest()
xgb03_selection = {
    "schema_version": 1,
    "run_id": xgb03_manifest["run_id"],
    "stage": "validation_threshold_selection",
    "model": "xgb",
    "feature_version": xgb03_manifest["feature_version"],
    "split_version": xgb03_manifest["split_version"],
    "preprocessing_version": xgb03_manifest["preprocessing_version"],
    "model_protocol_version": xgb03_manifest[
        "model_protocol_version"
    ],
    "prediction_partition": "validation",
    "prediction_file": xgb03_prediction_path.name,
    "prediction_sha256": xgb03_prediction_hash,
    "validation_record_id_sha256": xgb03_validation_id_hash,
    "fitted_pipeline_file": xgb03_model_path.name,
    "fitted_pipeline_sha256": xgb03_model_hash,
    "candidate_file": xgb03_candidates_path.name,
    "candidate_sha256": hashlib.sha256(
        xgb03_candidates_csv.encode("utf-8")
    ).hexdigest(),
    "candidate_grid": {
        "start": 0.001,
        "stop": 0.999,
        "step": 0.001,
        "inclusive": True,
        "count": 999,
    },
    "prediction_rule": (
        "p_approved >= threshold -> Approve (y_approved=1)"
    ),
    "objective": "maximize_validation_balanced_accuracy",
    "tie_break": [
        "minimum_absolute_distance_to_0.5",
        "lower_threshold",
    ],
    "maximizer_count": int(len(xgb03_max_indices)),
    "maximizing_thresholds": [
        int(xgb03_grid[index]) / 1000
        for index in xgb03_max_indices
    ],
    "selected_threshold": (
        int(xgb03_grid[xgb03_selected_index]) / 1000
    ),
    "selected_metrics": {
        "balanced_accuracy": float(
            xgb03_balanced_accuracy[xgb03_selected_index]
        ),
        "recall_approved": float(
            xgb03_tp[xgb03_selected_index]
            / xgb03_n_approved
        ),
        "recall_denied": float(
            xgb03_tn[xgb03_selected_index]
            / xgb03_n_denied
        ),
    },
    "validation_counts": {
        "rows": int(len(xgb03_predictions)),
        "n_approved": xgb03_n_approved,
        "n_denied": xgb03_n_denied,
        "tp_approved": int(xgb03_tp[xgb03_selected_index]),
        "fn_approved": int(xgb03_fn[xgb03_selected_index]),
        "tn_denied": int(xgb03_tn[xgb03_selected_index]),
        "fp_denied": int(xgb03_fp[xgb03_selected_index]),
    },
    "evaluation_status": (
        "experimental_feature_v0_2_validation_selection_only"
    ),
}
xgb03_selection_json = (
    json.dumps(xgb03_selection, ensure_ascii=False, indent=2)
    + "\n"
)

if xgb03_candidates_path.exists() or xgb03_selection_path.exists():
    if not (
        xgb03_candidates_path.is_file()
        and xgb03_selection_path.is_file()
    ):
        raise FileExistsError(
            "Partial XGB-03 artifacts exist; refusing to overwrite."
        )
    if (
        xgb03_candidates_path.read_bytes()
        != xgb03_candidates_csv.encode("utf-8")
    ):
        raise FileExistsError(
            "Existing XGB-03 threshold candidates differ."
        )
    if (
        xgb03_selection_path.read_text(encoding="utf-8")
        != xgb03_selection_json
    ):
        raise FileExistsError(
            "Existing XGB-03 threshold selection differs."
        )
    print("Existing XGB-03 artifacts match; no overwrite.")
else:
    xgb03_candidates_path.write_text(
        xgb03_candidates_csv,
        encoding="utf-8",
        newline="",
    )
    xgb03_selection_path.write_text(
        xgb03_selection_json,
        encoding="utf-8",
        newline="",
    )
    print("Created XGB-03 threshold artifacts.")

print(
    f"Validation: {len(xgb03_predictions):,} rows; "
    f"selected threshold "
    f"{xgb03_selection['selected_threshold']:.3f}; "
    f"Balanced Accuracy "
    f"{xgb03_selection['selected_metrics']['balanced_accuracy']:.6f}; "
    f"{xgb03_selection['maximizer_count']} maximizer(s)."
)


Existing XGB-03 artifacts match; no overwrite.
Validation: 43,995 rows; selected threshold 0.744; Balanced Accuracy 0.784547; 1 maximizer(s).


## XGB-03 result boundary

XGB-03 kết thúc tại threshold đã khóa từ Validation và bảng 999 ứng viên. Không dự báo hoặc đánh giá Test; XGB-04 chỉ được bắt đầu sau khi hai artifact threshold đã được kiểm tra.


## XGB-04 — Dự báo Test đúng một lần

Dùng fitted XGBoost pipeline của XGB-02 và threshold 0.744 đã khóa ở XGB-03. Kiểm tra feature, split, preprocessing, model và hash trước khi dự báo. Bắt buộc dùng lại bộ 1.000 stratified bootstrap indices chung ở cấp Dataset B; không tạo lại indices. Khi Test prediction đã tồn tại, cell chỉ kiểm tra và dùng lại, không gọi `predict_proba` lần hai. Chỉ chạy sau khi workload RF đã dừng.


In [10]:
from pathlib import Path
import hashlib
import json
import time

import joblib
import numpy as np
import pandas as pd
import yaml
from src.dataset_b.preprocessing import DatasetBSplitLoader

xgb04_root = PROJECT_ROOT
xgb04_shared_dir = xgb04_root / "artifacts/dataset_b"
xgb04_run_dir = xgb04_shared_dir / "xgb_baseline_v1"
xgb04_model_path = xgb04_run_dir / "xgb_full_train.joblib"
xgb04_manifest_path = xgb04_run_dir / "run_manifest.json"
xgb04_threshold_path = xgb04_run_dir / "threshold_selection.json"
xgb04_validation_path = xgb04_run_dir / "validation_predictions.csv"
xgb04_candidates_path = xgb04_run_dir / "threshold_candidates.csv"
xgb04_prediction_path = xgb04_run_dir / "test_predictions.csv"
xgb04_indices_path = xgb04_shared_dir / "rq1_test_bootstrap_indices.npz"
xgb04_indices_manifest_path = xgb04_shared_dir / "rq1_test_bootstrap_manifest.json"
xgb04_split_manifest_path = xgb04_shared_dir / "split_v1.json"

for xgb04_required_path in (
    xgb04_model_path,
    xgb04_manifest_path,
    xgb04_threshold_path,
    xgb04_validation_path,
    xgb04_candidates_path,
    xgb04_indices_path,
    xgb04_indices_manifest_path,
    xgb04_split_manifest_path,
):
    if not xgb04_required_path.is_file():
        raise FileNotFoundError(xgb04_required_path)

with xgb04_manifest_path.open(encoding="utf-8") as handle:
    xgb04_run_manifest = json.load(handle)
with xgb04_threshold_path.open(encoding="utf-8") as handle:
    xgb04_threshold_selection = json.load(handle)
with xgb04_indices_manifest_path.open(encoding="utf-8") as handle:
    xgb04_indices_manifest = json.load(handle)
with xgb04_split_manifest_path.open(encoding="utf-8") as handle:
    xgb04_split_manifest = json.load(handle)
with (xgb04_root / "config/dataset_b/model_protocol_v1.yaml").open(encoding="utf-8") as handle:
    xgb04_protocol = yaml.safe_load(handle)
with (xgb04_root / "config/dataset_b/feature_set_v1.yaml").open(encoding="utf-8") as handle:
    xgb04_feature_contract = yaml.safe_load(handle)

if xgb04_run_manifest["run_id"] != "xgb_baseline_v1":
    raise ValueError("Unexpected XGBoost run ID.")
if xgb04_run_manifest["stage"] != "full_train_refit_validation_predictions":
    raise ValueError("XGB-02 stage is not complete.")
if xgb04_run_manifest["model"] != "xgb":
    raise ValueError("Unexpected model in the run manifest.")
if xgb04_threshold_selection["run_id"] != xgb04_run_manifest["run_id"]:
    raise ValueError("Threshold and run IDs differ.")
if xgb04_threshold_selection["stage"] != "validation_threshold_selection":
    raise ValueError("XGB-03 threshold stage is not complete.")
if xgb04_threshold_selection["model"] != "xgb":
    raise ValueError("Unexpected model in threshold selection.")
if xgb04_threshold_selection["evaluation_status"] != "experimental_feature_v0_2_validation_selection_only":
    raise ValueError("XGB-03 threshold selection status differs.")
for xgb04_key in (
    "feature_version",
    "split_version",
    "preprocessing_version",
    "model_protocol_version",
):
    if xgb04_run_manifest[xgb04_key] != xgb04_threshold_selection[xgb04_key]:
        raise ValueError(f"Threshold lineage differs: {xgb04_key}.")
if xgb04_run_manifest["feature_version"] != xgb04_feature_contract["feature_version"]:
    raise ValueError("Feature version differs from the current contract.")
if xgb04_run_manifest["raw_feature_columns"] != xgb04_feature_contract["model_input_raw_columns"]:
    raise ValueError("Raw model inputs differ from the feature contract.")
if xgb04_run_manifest["split_version"] != "split_v1":
    raise ValueError("Unexpected split version.")
if xgb04_split_manifest["split_version"] != xgb04_run_manifest["split_version"]:
    raise ValueError("Split manifest version differs from the run.")
if xgb04_run_manifest["model_protocol_version"] != xgb04_protocol["protocol_version"]:
    raise ValueError("Model protocol version differs.")
if xgb04_run_manifest["preprocessing_version"] != xgb04_protocol["contracts"]["preprocessing_version"]:
    raise ValueError("Preprocessing version differs from the protocol.")
if xgb04_run_manifest["model_parameters"] != xgb04_protocol["models"]["xgb"]["parameters"]:
    raise ValueError("XGBoost parameters differ from the fixed protocol.")
if xgb04_protocol["test"]["use"] != "final_rq1_evaluation_only":
    raise ValueError("Test has an unexpected protocol role.")
if xgb04_protocol["test"]["may_change_preprocessing_model_or_threshold"] is not False:
    raise ValueError("Test must not change preprocessing, model, or threshold.")
xgb04_threshold = float(xgb04_threshold_selection["selected_threshold"])
if xgb04_threshold != 0.744:
    raise ValueError("XGB-03 locked threshold differs from 0.744.")
if hashlib.sha256(xgb04_validation_path.read_bytes()).hexdigest() != xgb04_threshold_selection["prediction_sha256"]:
    raise ValueError("Validation prediction hash differs from XGB-03.")
if hashlib.sha256(xgb04_candidates_path.read_bytes()).hexdigest() != xgb04_threshold_selection["candidate_sha256"]:
    raise ValueError("Threshold candidate hash differs from XGB-03.")
xgb04_model_hash = hashlib.sha256(xgb04_model_path.read_bytes()).hexdigest()
xgb04_threshold_hash = hashlib.sha256(xgb04_threshold_path.read_bytes()).hexdigest()
if xgb04_model_hash != xgb04_run_manifest["artifact_sha256"]["fitted_pipeline_sha256"]:
    raise ValueError("Fitted model hash differs from the run manifest.")
if xgb04_model_hash != xgb04_threshold_selection["fitted_pipeline_sha256"]:
    raise ValueError("Fitted model hash differs from XGB-03.")

xgb04_test = DatasetBSplitLoader().load("test")
if len(xgb04_test.X) != 43_996:
    raise ValueError("Expected 43,996 Test rows.")
if list(xgb04_test.X.columns) != xgb04_run_manifest["raw_feature_columns"]:
    raise ValueError("Test raw feature order differs from XGB-02.")
if len(xgb04_test.X.columns) != 12:
    raise ValueError("Expected 12 raw Test features.")
if set(xgb04_test.X.columns) & set(xgb04_test.audit.columns):
    raise ValueError("Audit columns appear in Test X.")
xgb04_ids = xgb04_test.record_ids.to_numpy(dtype=str)
xgb04_y = xgb04_test.y.to_numpy(dtype=np.int8)
if len(np.unique(xgb04_ids)) != len(xgb04_ids):
    raise ValueError("Test record IDs are not unique.")
if set(np.unique(xgb04_y)) != {0, 1}:
    raise ValueError("Test must contain both target classes.")
if xgb04_ids.tolist() != xgb04_split_manifest["test_record_ids"]:
    raise ValueError("Test record order differs from the split manifest.")
xgb04_ids_hash = hashlib.sha256(("\n".join(xgb04_ids) + "\n").encode("utf-8")).hexdigest()
if xgb04_indices_manifest["dataset"] != "dataset_b":
    raise ValueError("Shared bootstrap dataset differs.")
if xgb04_indices_manifest["partition"] != "test":
    raise ValueError("Shared bootstrap partition differs.")
if xgb04_indices_manifest["test_record_id_sha256"] != xgb04_ids_hash:
    raise ValueError("Shared bootstrap Test IDs differ.")
if xgb04_indices_manifest["split_version"] != "split_v1":
    raise ValueError("Shared bootstrap split version differs.")
if xgb04_indices_manifest["split_manifest_sha256"] != hashlib.sha256(xgb04_split_manifest_path.read_bytes()).hexdigest():
    raise ValueError("Shared bootstrap split manifest hash differs.")
if xgb04_indices_manifest["replicates"] != 1000 or xgb04_indices_manifest["seed"] != 42:
    raise ValueError("Shared bootstrap replicate count or seed differs.")
if xgb04_indices_manifest["shape"] != [1000, len(xgb04_y)]:
    raise ValueError("Shared bootstrap shape differs.")
if xgb04_indices_manifest["dtype"] != "int32":
    raise ValueError("Shared bootstrap dtype differs.")
if xgb04_indices_manifest["class_counts"] != {"0": int((xgb04_y == 0).sum()), "1": int((xgb04_y == 1).sum())}:
    raise ValueError("Shared bootstrap class counts differ.")
if "xgboost" not in xgb04_indices_manifest["reuse"]:
    raise ValueError("Shared bootstrap is not approved for XGBoost reuse.")
if hashlib.sha256(xgb04_indices_path.read_bytes()).hexdigest() != xgb04_indices_manifest["indices_sha256"]:
    raise ValueError("Shared bootstrap index hash differs.")

if xgb04_prediction_path.exists():
    xgb04_inference = xgb04_run_manifest.get("test_inference")
    if xgb04_inference is None:
        raise FileExistsError("Test prediction exists without inference metadata; refusing a second inference.")
    if xgb04_inference["partition"] != "test" or xgb04_inference["rows"] != len(xgb04_y):
        raise ValueError("Saved Test inference partition or row count differs.")
    if xgb04_inference["prediction_file"] != xgb04_prediction_path.name:
        raise ValueError("Saved Test inference prediction filename differs.")
    xgb04_predictions = pd.read_csv(xgb04_prediction_path, dtype={"record_id": str})
    if hashlib.sha256(xgb04_prediction_path.read_bytes()).hexdigest() != xgb04_inference["prediction_sha256"]:
        raise ValueError("Saved Test prediction hash differs.")
    if xgb04_inference["model_sha256"] != xgb04_model_hash:
        raise ValueError("Saved Test inference model hash differs.")
    if xgb04_inference["threshold_sha256"] != xgb04_threshold_hash:
        raise ValueError("Saved Test inference threshold hash differs.")
    if xgb04_inference["bootstrap_indices_sha256"] != xgb04_indices_manifest["indices_sha256"]:
        raise ValueError("Saved Test inference bootstrap hash differs.")
    if xgb04_inference["selected_threshold"] != xgb04_threshold:
        raise ValueError("Saved Test inference threshold differs.")
    if xgb04_inference["predict_proba_calls_on_test"] != 1:
        raise ValueError("Test predict_proba call count differs from one.")
    print("Reusing saved XGBoost Test predictions; no model inference.")
else:
    if "test_inference" in xgb04_run_manifest:
        raise FileExistsError("Inference metadata exists without Test prediction file.")
    xgb04_pipeline = joblib.load(xgb04_model_path)
    xgb04_estimator = xgb04_pipeline.named_steps["estimator"]
    if type(xgb04_estimator).__name__ != "XGBClassifier":
        raise TypeError("Saved estimator is not XGBClassifier.")
    if not np.array_equal(xgb04_estimator.classes_, np.array([0, 1])):
        raise ValueError("Saved estimator class order differs from [0, 1].")
    if int(xgb04_estimator.n_features_in_) != xgb04_run_manifest["encoded_feature_count"]:
        raise ValueError("Saved estimator encoded feature count differs.")
    if len(xgb04_pipeline.named_steps["preprocess"].get_feature_names_out()) != 66:
        raise ValueError("Saved preprocessing output count differs from 66.")
    for xgb04_key, xgb04_value in xgb04_run_manifest["model_parameters"].items():
        if xgb04_estimator.get_params()[xgb04_key] != xgb04_value:
            raise ValueError(f"Saved XGBoost parameter differs: {xgb04_key}.")
    xgb04_start = time.perf_counter()
    xgb04_p = np.asarray(
        xgb04_pipeline.predict_proba(xgb04_test.X)[:, 1],
        dtype=np.float64,
    )
    xgb04_inference_seconds = time.perf_counter() - xgb04_start
    if len(xgb04_p) != len(xgb04_y):
        raise ValueError("Test probability length differs from Test rows.")
    if not np.isfinite(xgb04_p).all() or np.any((xgb04_p < 0) | (xgb04_p > 1)):
        raise ValueError("Test probabilities are non-finite or outside [0, 1].")
    xgb04_predicted = (xgb04_p >= xgb04_threshold).astype(np.int8)
    xgb04_predictions = pd.DataFrame({
        "record_id": xgb04_ids,
        "y_approved": xgb04_y,
        "p_approved": xgb04_p,
        "p_denied": 1.0 - xgb04_p,
        "predicted_approved": xgb04_predicted,
        "predicted_denied": 1 - xgb04_predicted,
    })
    xgb04_predictions.to_csv(xgb04_prediction_path, index=False, float_format="%.17g")
    xgb04_run_manifest["test_inference"] = {
        "partition": "test",
        "rows": len(xgb04_y),
        "model_sha256": xgb04_model_hash,
        "threshold_sha256": xgb04_threshold_hash,
        "selected_threshold": xgb04_threshold,
        "prediction_file": xgb04_prediction_path.name,
        "prediction_sha256": hashlib.sha256(xgb04_prediction_path.read_bytes()).hexdigest(),
        "bootstrap_indices_file": xgb04_indices_path.name,
        "bootstrap_indices_sha256": xgb04_indices_manifest["indices_sha256"],
        "inference_seconds_including_preprocessing": xgb04_inference_seconds,
        "predict_proba_calls_on_test": 1,
    }
    xgb04_manifest_path.write_text(
        json.dumps(xgb04_run_manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        newline="",
    )
    print(f"Saved one XGBoost Test prediction pass: {len(xgb04_y):,} rows in {xgb04_inference_seconds:.3f}s.")

xgb04_expected_columns = [
    "record_id",
    "y_approved",
    "p_approved",
    "p_denied",
    "predicted_approved",
    "predicted_denied",
]
if list(xgb04_predictions.columns) != xgb04_expected_columns:
    raise ValueError("Unexpected Test prediction columns.")
if not np.array_equal(xgb04_predictions["record_id"].to_numpy(dtype=str), xgb04_ids):
    raise ValueError("Saved Test prediction ID order differs.")
if not np.array_equal(xgb04_predictions["y_approved"].to_numpy(dtype=np.int8), xgb04_y):
    raise ValueError("Saved Test labels differ.")
xgb04_saved_p = xgb04_predictions["p_approved"].to_numpy(dtype=float)
if not np.isfinite(xgb04_saved_p).all() or np.any((xgb04_saved_p < 0) | (xgb04_saved_p > 1)):
    raise ValueError("Saved Test probabilities are invalid.")
# XGBoost probabilities already persisted by the first run are float32.
xgb04_probability_atol = np.finfo(np.float32).eps / 2
if not np.allclose(xgb04_predictions["p_denied"], 1.0 - xgb04_saved_p, rtol=0, atol=xgb04_probability_atol):
    raise ValueError("Saved denied probabilities differ from 1 - p_approved.")
if not np.array_equal(xgb04_predictions["predicted_approved"].to_numpy(dtype=np.int8), (xgb04_saved_p >= xgb04_threshold).astype(np.int8)):
    raise ValueError("Saved Approved decisions differ from the locked threshold.")
if not np.array_equal(xgb04_predictions["predicted_denied"].to_numpy(dtype=np.int8), (xgb04_saved_p < xgb04_threshold).astype(np.int8)):
    raise ValueError("Saved Denied decisions differ from the locked threshold.")


Reusing saved XGBoost Test predictions; no model inference.


## XGB-04 — RQ1 point metrics và 95% bootstrap CI

Tính cùng metric/calibration contract với LR-04: ROC-AUC trên P(Approved), average precision cho Denied, các metric phân lớp tại threshold cố định, Brier, calibration và thời gian. Calibration curve dùng 10 quantile bins; intercept là logistic offset với slope bằng 1, còn slope từ logistic recalibration có intercept tự do. CI percentile 2.5% và 97.5% từ đúng 1.000 shared bootstrap replicates. Invalid metric được ghi theo replicate và loại khỏi CI của chính metric đó, không resample. Curve và thời gian không nhận CI từ bootstrap prediction. Nhãn kết quả là `xgb_baseline_v1 / experimental feature lock v0.2`, không phải FINAL_MODEL_FEATURES.


In [11]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize, root
from scipy.special import expit, logit
from sklearn.metrics import average_precision_score, roc_auc_score

xgb04_run_dir = PROJECT_ROOT / "artifacts/dataset_b/xgb_baseline_v1"
xgb04_shared_dir = PROJECT_ROOT / "artifacts/dataset_b"
xgb04_manifest_path = xgb04_run_dir / "run_manifest.json"
xgb04_prediction_path = xgb04_run_dir / "test_predictions.csv"
xgb04_metrics_path = xgb04_run_dir / "rq1_test_metrics.json"
xgb04_bootstrap_path = xgb04_run_dir / "rq1_bootstrap_metrics.csv"
xgb04_indices_path = xgb04_shared_dir / "rq1_test_bootstrap_indices.npz"
xgb04_indices_manifest_path = xgb04_shared_dir / "rq1_test_bootstrap_manifest.json"
xgb04_evaluation_label = "xgb_baseline_v1 / experimental feature lock v0.2"

for xgb04_required_path in (
    xgb04_manifest_path,
    xgb04_prediction_path,
    xgb04_indices_path,
    xgb04_indices_manifest_path,
):
    if not xgb04_required_path.is_file():
        raise FileNotFoundError(xgb04_required_path)
with xgb04_manifest_path.open(encoding="utf-8") as handle:
    xgb04_run_manifest = json.load(handle)
with xgb04_indices_manifest_path.open(encoding="utf-8") as handle:
    xgb04_indices_manifest = json.load(handle)
xgb04_inference = xgb04_run_manifest.get("test_inference")
if xgb04_inference is None:
    raise RuntimeError("XGB-04 Test inference must complete before evaluation.")
xgb04_prediction_sha = hashlib.sha256(xgb04_prediction_path.read_bytes()).hexdigest()
if xgb04_prediction_sha != xgb04_inference["prediction_sha256"]:
    raise ValueError("Test prediction hash differs from the run manifest.")
if hashlib.sha256(xgb04_indices_path.read_bytes()).hexdigest() != xgb04_indices_manifest["indices_sha256"]:
    raise ValueError("Shared bootstrap index hash differs.")
if xgb04_inference["bootstrap_indices_sha256"] != xgb04_indices_manifest["indices_sha256"]:
    raise ValueError("Inference bootstrap hash differs from the shared manifest.")
if xgb04_inference["predict_proba_calls_on_test"] != 1:
    raise ValueError("Test predict_proba call count differs from one.")

xgb04_predictions = pd.read_csv(xgb04_prediction_path, dtype={"record_id": str})
xgb04_expected_columns = [
    "record_id",
    "y_approved",
    "p_approved",
    "p_denied",
    "predicted_approved",
    "predicted_denied",
]
if list(xgb04_predictions.columns) != xgb04_expected_columns:
    raise ValueError("Unexpected Test prediction columns.")
if len(xgb04_predictions) != xgb04_inference["rows"]:
    raise ValueError("Test prediction count differs from inference metadata.")
if len(xgb04_predictions) != xgb04_indices_manifest["test_rows"] or len(xgb04_predictions) != 43_996:
    raise ValueError("Test prediction count differs from the shared bootstrap contract.")
if not xgb04_predictions["record_id"].is_unique:
    raise ValueError("Test prediction record IDs are not unique.")
xgb04_y = xgb04_predictions["y_approved"].to_numpy(dtype=np.int8)
xgb04_p = xgb04_predictions["p_approved"].to_numpy(dtype=np.float64)
xgb04_threshold = float(xgb04_inference["selected_threshold"])
if xgb04_threshold != 0.744:
    raise ValueError("Test inference threshold differs from XGB-03.")
if set(np.unique(xgb04_y)) != {0, 1}:
    raise ValueError("Test predictions must contain both target classes.")
if not np.isfinite(xgb04_p).all() or np.any((xgb04_p < 0) | (xgb04_p > 1)):
    raise ValueError("Test probabilities are non-finite or outside [0, 1].")
# Accept the rounding bound of float32 XGBoost probabilities after CSV reload.
xgb04_probability_atol = np.finfo(np.float32).eps / 2
if not np.allclose(xgb04_predictions["p_denied"], 1.0 - xgb04_p, rtol=0, atol=xgb04_probability_atol):
    raise ValueError("Denied probabilities differ from 1 - p_approved.")
if not np.array_equal(xgb04_predictions["predicted_approved"].to_numpy(dtype=np.int8), (xgb04_p >= xgb04_threshold).astype(np.int8)):
    raise ValueError("Approved decisions differ from the locked threshold.")
if not np.array_equal(xgb04_predictions["predicted_denied"].to_numpy(dtype=np.int8), (xgb04_p < xgb04_threshold).astype(np.int8)):
    raise ValueError("Denied decisions differ from the locked threshold.")
xgb04_ids_sha = hashlib.sha256(("\n".join(xgb04_predictions["record_id"]) + "\n").encode("utf-8")).hexdigest()
if xgb04_ids_sha != xgb04_indices_manifest["test_record_id_sha256"]:
    raise ValueError("Test prediction ID hash differs from shared bootstrap.")

xgb04_scalar_names = [
    "roc_auc_approved",
    "average_precision_denied",
    "precision_denied",
    "recall_denied",
    "f1_denied",
    "recall_approved",
    "balanced_accuracy",
    "brier_approved",
    "calibration_intercept",
    "calibration_slope",
]

def xgb04_calibration(y, p):
    z = logit(np.clip(p, 1e-15, 1 - 1e-15))
    target = y.astype(np.float64)
    invalid = {}
    intercept = np.nan
    slope = np.nan
    try:
        intercept = float(brentq(
            lambda a: float(np.mean(expit(z + a) - target)),
            -80,
            80,
            xtol=1e-10,
        ))
    except (ArithmeticError, ValueError, RuntimeError) as exc:
        invalid["calibration_intercept"] = str(exc)

    def loss_and_gradient(theta):
        eta = theta[0] + theta[1] * z
        error = expit(eta) - target
        loss = float(np.mean(np.logaddexp(0, eta) - target * eta))
        gradient = np.array(
            [np.mean(error), np.mean(error * z)],
            dtype=np.float64,
        )
        return loss, gradient

    try:
        slope_fit = minimize(
            loss_and_gradient,
            x0=np.array([0.0, 1.0]),
            method="L-BFGS-B",
            jac=True,
            options={"maxiter": 100, "ftol": 1e-12, "gtol": 1e-9},
        )
        slope_parameters = slope_fit.x
        if not slope_fit.success or not np.isfinite(slope_parameters).all():
            initial = slope_parameters if np.isfinite(slope_parameters).all() else np.array([0.0, 1.0])
            fallback = root(
                lambda theta: loss_and_gradient(theta)[1],
                initial,
                method="hybr",
                tol=1e-10,
            )
            slope_parameters = fallback.x
        score = loss_and_gradient(slope_parameters)[1]
        if not np.isfinite(slope_parameters).all() or not np.isfinite(score).all() or np.max(np.abs(score)) > 1e-7:
            raise ArithmeticError("Calibration slope score did not converge.")
        slope = float(slope_parameters[1])
    except (ArithmeticError, ValueError, RuntimeError) as exc:
        invalid["calibration_slope"] = str(exc)
    return intercept, slope, invalid

def xgb04_metrics(y, p, threshold):
    invalid = {}
    values = {}
    predicted_approved = p >= threshold
    approved = y == 1
    denied = y == 0
    tp = int(np.count_nonzero(approved & predicted_approved))
    fn = int(np.count_nonzero(approved & ~predicted_approved))
    tn = int(np.count_nonzero(denied & ~predicted_approved))
    fp = int(np.count_nonzero(denied & predicted_approved))
    n_approved = tp + fn
    n_denied = tn + fp
    if n_approved == 0 or n_denied == 0:
        raise ArithmeticError("Both true classes are required.")
    values["roc_auc_approved"] = float(roc_auc_score(y, p))
    values["average_precision_denied"] = float(average_precision_score(1 - y, 1 - p))
    if tn + fn:
        values["precision_denied"] = tn / (tn + fn)
    else:
        values["precision_denied"] = np.nan
        invalid["precision_denied"] = "no_predicted_denied"
    values["recall_denied"] = tn / n_denied
    values["f1_denied"] = 2 * tn / (2 * tn + fn + fp)
    values["recall_approved"] = tp / n_approved
    values["balanced_accuracy"] = (values["recall_approved"] + values["recall_denied"]) / 2
    values["brier_approved"] = float(np.mean((p - y) ** 2))
    values["calibration_intercept"], values["calibration_slope"], calibration_invalid = xgb04_calibration(y, p)
    invalid.update(calibration_invalid)
    for name, value in values.items():
        if not np.isfinite(value):
            invalid.setdefault(name, "nonfinite_metric")
    return values, invalid, {
        "tp_approved": tp,
        "fn_approved": fn,
        "tn_denied": tn,
        "fp_denied": fp,
        "n_approved": n_approved,
        "n_denied": n_denied,
    }

def xgb04_curve(y, p):
    edges = np.unique(np.quantile(p, np.linspace(0, 1, 11)))
    if len(edges) < 2:
        raise ArithmeticError("Calibration curve has fewer than two distinct edges.")
    bin_id = np.searchsorted(edges[1:-1], p, side="right")
    bins = []
    for i in range(len(edges) - 1):
        selected = bin_id == i
        if not selected.any():
            continue
        bins.append({
            "bin": i,
            "lower_edge": float(edges[i]),
            "upper_edge": float(edges[i + 1]),
            "count": int(selected.sum()),
            "mean_p_approved": float(p[selected].mean()),
            "observed_approval_rate": float(y[selected].mean()),
        })
    if sum(item["count"] for item in bins) != len(y):
        raise ArithmeticError("Calibration bins do not cover all Test rows.")
    return bins

if xgb04_metrics_path.exists() or xgb04_bootstrap_path.exists():
    if not (xgb04_metrics_path.is_file() and xgb04_bootstrap_path.is_file()):
        raise FileExistsError("Partial XGB-04 metric artifacts exist.")
    xgb04_evaluation = xgb04_run_manifest.get("rq1_test_evaluation")
    if xgb04_evaluation is None:
        raise FileExistsError("Metric artifacts exist without run manifest references.")
    if xgb04_evaluation.get("evaluation_label") != xgb04_evaluation_label:
        raise ValueError("Saved XGB-04 evaluation label differs.")
    if hashlib.sha256(xgb04_metrics_path.read_bytes()).hexdigest() != xgb04_evaluation["metrics_sha256"]:
        raise ValueError("Saved XGB-04 metric hash differs.")
    if hashlib.sha256(xgb04_bootstrap_path.read_bytes()).hexdigest() != xgb04_evaluation["bootstrap_metrics_sha256"]:
        raise ValueError("Saved XGB-04 bootstrap metric hash differs.")
    with xgb04_metrics_path.open(encoding="utf-8") as handle:
        xgb04_saved_metrics = json.load(handle)
    if xgb04_saved_metrics["evaluation_label"] != xgb04_evaluation_label:
        raise ValueError("Saved XGB-04 metric payload label differs.")
    if xgb04_saved_metrics["prediction_sha256"] != xgb04_prediction_sha:
        raise ValueError("Saved XGB-04 metric prediction hash differs.")
    if xgb04_saved_metrics["model_sha256"] != xgb04_inference["model_sha256"]:
        raise ValueError("Saved XGB-04 metric model hash differs.")
    if xgb04_saved_metrics["bootstrap_indices_sha256"] != xgb04_indices_manifest["indices_sha256"]:
        raise ValueError("Saved XGB-04 metric bootstrap index hash differs.")
    if xgb04_saved_metrics["threshold"] != xgb04_threshold:
        raise ValueError("Saved XGB-04 metric threshold differs.")
    xgb04_saved_bootstrap = pd.read_csv(xgb04_bootstrap_path)
    if len(xgb04_saved_bootstrap) != 1000:
        raise ValueError("Saved XGB-04 bootstrap metric row count differs.")
    if list(xgb04_saved_bootstrap.columns) != ["replicate_id", *xgb04_scalar_names, "invalid_metrics"]:
        raise ValueError("Saved XGB-04 bootstrap metric columns differ.")
    print("Existing XGB-04 metrics match; no recomputation.")
else:
    xgb04_point, xgb04_point_invalid, xgb04_counts = xgb04_metrics(
        xgb04_y,
        xgb04_p,
        xgb04_threshold,
    )
    if xgb04_point_invalid:
        raise ArithmeticError(f"Test point metrics are invalid: {xgb04_point_invalid}")
    xgb04_calibration_curve = xgb04_curve(xgb04_y, xgb04_p)
    with np.load(xgb04_indices_path, allow_pickle=False) as package:
        xgb04_indices = package["indices"]
    if xgb04_indices.shape != (1000, len(xgb04_y)):
        raise ValueError("Shared bootstrap matrix shape differs.")
    if xgb04_indices.dtype != np.int32:
        raise ValueError("Shared bootstrap matrix dtype differs from int32.")
    if not np.all((xgb04_indices >= 0) & (xgb04_indices < len(xgb04_y))):
        raise ValueError("Shared bootstrap matrix contains an invalid row index.")
    xgb04_n_denied = xgb04_counts["n_denied"]
    if not np.all(xgb04_y[xgb04_indices[:, :xgb04_n_denied]] == 0):
        raise ValueError("Shared bootstrap denied block differs.")
    if not np.all(xgb04_y[xgb04_indices[:, xgb04_n_denied:]] == 1):
        raise ValueError("Shared bootstrap approved block differs.")
    xgb04_rows = []
    for xgb04_replicate_id, xgb04_row_indices in enumerate(xgb04_indices):
        xgb04_sampled, xgb04_invalid, _ = xgb04_metrics(
            xgb04_y[xgb04_row_indices],
            xgb04_p[xgb04_row_indices],
            xgb04_threshold,
        )
        xgb04_rows.append({
            "replicate_id": xgb04_replicate_id,
            **xgb04_sampled,
            "invalid_metrics": json.dumps(xgb04_invalid, ensure_ascii=False, sort_keys=True),
        })
        if (xgb04_replicate_id + 1) % 100 == 0:
            print(f"Bootstrap {xgb04_replicate_id + 1}/1000", flush=True)
    xgb04_bootstrap_frame = pd.DataFrame(
        xgb04_rows,
        columns=["replicate_id", *xgb04_scalar_names, "invalid_metrics"],
    )
    if len(xgb04_bootstrap_frame) != 1000:
        raise ValueError("Expected 1,000 bootstrap metric rows.")
    xgb04_ci = {}
    for xgb04_name in xgb04_scalar_names:
        xgb04_values = xgb04_bootstrap_frame[xgb04_name].to_numpy(dtype=np.float64)
        xgb04_valid = xgb04_values[np.isfinite(xgb04_values)]
        xgb04_ci[xgb04_name] = {
            "lower_2_5": float(np.quantile(xgb04_valid, 0.025, method="linear")) if len(xgb04_valid) else None,
            "upper_97_5": float(np.quantile(xgb04_valid, 0.975, method="linear")) if len(xgb04_valid) else None,
            "valid_replicates": int(len(xgb04_valid)),
            "invalid_replicates": int(len(xgb04_values) - len(xgb04_valid)),
        }
    xgb04_bootstrap_csv = xgb04_bootstrap_frame.to_csv(index=False, float_format="%.17g")
    xgb04_metrics_payload = {
        "schema_version": 1,
        "run_id": xgb04_run_manifest["run_id"],
        "evaluation_label": xgb04_evaluation_label,
        "evaluation_status": "complete_rq1_evaluation_experimental_feature_v0_2" if all(item["valid_replicates"] >= 2 for item in xgb04_ci.values()) else "partial_ci",
        "feature_version": xgb04_run_manifest["feature_version"],
        "split_version": xgb04_run_manifest["split_version"],
        "preprocessing_version": xgb04_run_manifest["preprocessing_version"],
        "model_protocol_version": xgb04_run_manifest["model_protocol_version"],
        "partition": "test",
        "rows": len(xgb04_y),
        "threshold": xgb04_threshold,
        "prediction_file": xgb04_prediction_path.name,
        "prediction_sha256": xgb04_prediction_sha,
        "model_sha256": xgb04_inference["model_sha256"],
        "bootstrap_indices_file": xgb04_indices_path.name,
        "bootstrap_indices_sha256": xgb04_indices_manifest["indices_sha256"],
        "bootstrap_replicates_file": xgb04_bootstrap_path.name,
        "bootstrap_replicates_sha256": hashlib.sha256(xgb04_bootstrap_csv.encode("utf-8")).hexdigest(),
        "counts": xgb04_counts,
        "point_metrics": xgb04_point,
        "confidence_intervals_95_percentile": xgb04_ci,
        "calibration_curve": xgb04_calibration_curve,
        "computational_metrics_seconds": {
            "full_train_fit": xgb04_run_manifest["fit_seconds"],
            "test_inference_including_preprocessing": xgb04_inference["inference_seconds_including_preprocessing"],
        },
        "methods": {
            "roc_auc_approved": "sklearn roc_auc_score(y_approved, p_approved)",
            "average_precision_denied": "sklearn average_precision_score(1-y_approved, 1-p_approved)",
            "predicted_approved": "p_approved >= locked threshold",
            "brier_approved": "mean((p_approved-y_approved)^2)",
            "calibration_curve": "10 quantile bins from Test p_approved; duplicate edges collapsed; no bootstrap CI",
            "calibration_intercept": "logistic offset recalibration of y_approved on logit(clip(p,1e-15,1-1e-15)); slope fixed at 1",
            "calibration_slope": "unpenalized logistic recalibration of y_approved on logit(clip(p,1e-15,1-1e-15)); free intercept and slope",
            "bootstrap": "1000 shared stratified true-class row-index replicates, seed 42, model and threshold fixed; 2.5 and 97.5 empirical percentiles",
            "invalid_replicate": "record reason in invalid_metrics; exclude only the undefined scalar metric from its CI; do not resample",
            "timing_ci": "no prediction bootstrap CI for fit or inference time",
        },
    }
    xgb04_metrics_json = json.dumps(xgb04_metrics_payload, ensure_ascii=False, indent=2) + "\n"
    xgb04_bootstrap_path.write_text(xgb04_bootstrap_csv, encoding="utf-8", newline="")
    xgb04_metrics_path.write_text(xgb04_metrics_json, encoding="utf-8", newline="")
    xgb04_run_manifest["rq1_test_evaluation"] = {
        "status": xgb04_metrics_payload["evaluation_status"],
        "evaluation_label": xgb04_evaluation_label,
        "metrics_file": xgb04_metrics_path.name,
        "metrics_sha256": hashlib.sha256(xgb04_metrics_path.read_bytes()).hexdigest(),
        "bootstrap_metrics_file": xgb04_bootstrap_path.name,
        "bootstrap_metrics_sha256": hashlib.sha256(xgb04_bootstrap_path.read_bytes()).hexdigest(),
        "bootstrap_indices_file": xgb04_indices_path.name,
        "bootstrap_indices_sha256": xgb04_indices_manifest["indices_sha256"],
    }
    xgb04_manifest_path.write_text(
        json.dumps(xgb04_run_manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        newline="",
    )
    xgb04_invalid_total = sum(item["invalid_replicates"] for item in xgb04_ci.values())
    print(f"Saved XGB-04 metrics and 95% CIs. Invalid metric-replicate values: {xgb04_invalid_total}.")


Existing XGB-04 metrics match; no recomputation.


## XGB-04 — Quy ước kỹ thuật

**Phạm vi run.** Đây là đánh giá RQ1 của XGBoost baseline `xgb_baseline_v1` với feature lock v0.2 ở trạng thái experimental. Split là `split_v1`, preprocessing là `dataset_b_preprocessing_v2`, đầu vào gồm 12 raw features và pipeline XGBoost đã fit trên toàn Train tạo 66 cột sau mã hóa. Đây không phải FINAL_MODEL_FEATURES của toàn dự án.

**Ranh giới dữ liệu và dự báo.** Train dùng để fit; Validation dùng để chọn ngưỡng; Test gồm 43.996 hồ sơ chỉ được đưa qua fitted pipeline đúng một lượt `predict_proba`. Không fit lại, tuning lại, đổi feature, model hoặc ngưỡng theo Test. Dự báo lưu theo thứ tự `record_id` của split manifest. Nhãn `y_approved=1` là Approved, `y_approved=0` là Denied; `p_approved` là P(Approved), `p_denied=1-p_approved`. Ngưỡng đã khóa là 0,744: `p_approved >= 0.744` dự báo Approved, còn lại là Denied.

**Metric và calibration.** Dùng đúng contract LR-04: ROC-AUC trên `y_approved` và `p_approved`; average precision của Denied trên `1-y_approved` và `1-p_approved`; precision/recall/F1 Denied, Recall Approved, Balanced Accuracy, Brier Approved; calibration curve 10 quantile bins. Xác suất được clip vào `[1e-15, 1-1e-15]` trước logit. Calibration intercept là logistic offset với slope cố định 1; calibration slope là logistic recalibration không regularization với intercept tự do.

**Độ bất định.** Bắt buộc tái sử dụng ma trận 1.000 Test row indices chung của Dataset B cho LR/RF/XGBoost, stratified theo hai lớp thật, seed 42. Giữ nguyên model và threshold. CI 95% cho từng scalar predictive metric dùng percentile 2,5% và 97,5% với `method=linear`. Metric không xác định được ghi lý do và loại riêng khỏi CI tương ứng; không lấy mẫu bù. Curve và timing không có prediction-bootstrap CI.

**Artifact dự kiến khi chạy.** `test_predictions.csv`, `rq1_test_metrics.json`, `rq1_bootstrap_metrics.csv` và các liên kết hash tương ứng trong `run_manifest.json`. So sánh LR/RF/XGBoost theo cặp thuộc task riêng, không nằm trong XGB-04.


## XGB-04 result boundary

XGB-04 kết thúc tại Test predictions, RQ1 point metrics, calibration và 95% bootstrap CIs của riêng XGBoost. Không thực hiện paired model comparison trong notebook này.
